# 00 · Environment Setup & Data Ingestion

**Goal:** Establish a reproducible Colab workspace, configure secure connections (SSH/GitHub), and retrieve the raw dataset via DVC or Kaggle.

**Output:** A fully configured MLOps environment with a populated `data/raw/` directory, ready for downstream profiling and modeling.

---
| Section | Content |
|---|---|
| 0 | Bootstrap: Mount Drive, SSH Keys & Git Clone |
| 1 | Environment Init: Install Dependencies & DVC Pull |
| 2 | Data Ingestion: Kaggle Download & DVC Tracking (If needed) |

In [ ]:
# ===================================================================
# 1. BOOTSTRAP: Mount Drive, Setup SSH, and Clone Repo
# ===================================================================
import os
import subprocess
import sys

# A. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# B. Setup SSH Key from Drive
ssh_dir = os.path.expanduser('~/.ssh')
os.makedirs(ssh_dir, exist_ok=True)

key_source = '/content/drive/MyDrive/ssh_config/housing_key'
key_dest = os.path.join(ssh_dir, 'id_rsa')

if os.path.exists(key_source):
    subprocess.run(['cp', key_source, key_dest], check=True)
    subprocess.run(['chmod', '600', key_dest], check=True)
    
    # Add GitHub to known hosts to prevent interactive prompts
    with open(os.path.join(ssh_dir, 'known_hosts'), 'a') as f:
        subprocess.run(['ssh-keyscan', '-H', 'github.com'], stdout=f, stderr=subprocess.DEVNULL)
    print("✅ SSH Key configured successfully.")
else:
    print(f"❌ ERROR: SSH Key not found at {key_source}")

# C. Clone Repository
repo_url = "git@github.com:ebramrafat653-wq/california_housing_full_project.git"
repo_dir = "/content/california_housing_full_project"

if not os.path.exists(repo_dir):
    print("📥 Cloning repository...")
    subprocess.run(['git', 'clone', repo_url, repo_dir], check=True)
else:
    print("✅ Repository already exists.")

# D. Add to Python Path
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)
    print(f"✅ Added {repo_dir} to sys.path")

Mounted at /content/drive
✅ SSH Key configured successfully.
📥 Cloning repository...
✅ Added /content/california_housing_full_project to sys.path


In [2]:
# ===================================================================
# 2. INITIALIZE ENVIRONMENT & DVC PULL
# ===================================================================
import os
import sys
from pathlib import Path

# 1. Add repo path so Python can find the 'src' module
repo_path = Path("/content/california_housing_full_project")
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
os.chdir(repo_path)

# 2. Import setup functions 
# (NOTE: setup_drive_symlinks is REMOVED to avoid conflict with DVC)
from src.utils.colab_setup import install_dependencies, initialize_dvc, dvc_pull

print("🛠️ Testing Mode: Initializing environment...")

# 3. Install dependencies
install_dependencies(repo_path / "requirements.txt")

# 4. Initialize DVC
initialize_dvc(repo_path)

# 5. Pull data from DVC remote (Drive)
print("📥 Pulling data from DVC remote...")
dvc_pull()

print("✅ Environment initialized and data pulled successfully.")

🛠️ Testing Mode: Initializing environment...
2026-06-05 00:34:55 | INFO     | src.utils.colab_setup | Installing dependencies…


/content/california_housing_full_project/src/utils/paths.py:28: RuntimeWarning: Logger auto-initialized with defaults. Call setup_logging() explicitly at startup for full control.
  logger = get_logger(__name__)


2026-06-05 00:35:20 | INFO     | src.utils.colab_setup | Dependencies installed
2026-06-05 00:35:20 | INFO     | src.utils.colab_setup | ✅ DVC already initialized (config from GitHub)
📥 Pulling data from DVC remote...
2026-06-05 00:35:20 | INFO     | src.utils.colab_setup | DVC pull: ['data/raw']
2026-06-05 00:35:25 | INFO     | src.utils.colab_setup | ✅ DVC pull complete
✅ Environment initialized and data pulled successfully.


In [7]:
# ===================================================================
# 3. DATA INGESTION & DVC TRACKING
# ===================================================================
from src.data.ingestion import run_ingestion

print("🚀 Starting Data Ingestion Pipeline...")

report = run_ingestion(
    unzip=True,           
    force=True,          
    auto_track_dvc=True   
)

print(report.summary())

🚀 Starting Data Ingestion Pipeline...
2026-06-04 00:18:39 | INFO     | src.data.ingestion | ────────────────────────────────────────────────────────────
2026-06-04 00:18:39 | INFO     | src.data.ingestion |   Data ingestion started
2026-06-04 00:18:39 | INFO     | src.data.ingestion | ────────────────────────────────────────────────────────────
2026-06-04 00:18:39 | INFO     | src.data.ingestion | Config loaded: /content/california_housing_full_project/configs/data_config.yaml
2026-06-04 00:18:39 | INFO     | src.data.ingestion | Kaggle credentials installed: /root/.kaggle/kaggle.json
2026-06-04 00:18:39 | INFO     | src.data.ingestion | DVC initialized — trying pull first…
2026-06-04 00:18:39 | INFO     | src.data.ingestion | DVC pull: data/raw
2026-06-04 00:18:41 | INFO     | src.data.ingestion | ✅ Data ready via DVC (1 file(s))
2026-06-04 00:18:41 | INFO     | src.data.ingestion | ✅ Integrity check passed
2026-06-04 00:18:41 | INFO     | src.data.ingestion | Integrity: 1 file(s), 1.